# Customer Basket Analysis Project
A cleaned and structured version of your notebook with clear sections.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

## 2. Load Datasets

In [ ]:
basket = pd.read_csv('basket_details.csv')
customers = pd.read_csv('customer_details.csv')

print(basket.head())
print(customers.head())

## 3. Merge Data

In [ ]:
df_final = pd.merge(basket, customers, on='customer_id', how='inner')
print(df_final.head())
print(df_final.info())

## 4. Exploratory Analysis

In [ ]:
gender_analysis = df_final.groupby('sex')['basket_count'].sum()
print(gender_analysis)

bins = [0, 20, 30, 40, 50, 100]
labels = ['0-20', '21-30', '31-40', '41-50', '50+']
df_final['age_group'] = pd.cut(df_final['customer_age'], bins=bins, labels=labels)
age_analysis = df_final.groupby('age_group')['basket_count'].sum()
print(age_analysis)

## 5. Visualizations

In [ ]:
age_analysis.plot(kind='bar', edgecolor='black')
plt.title('Shopping Count by Age Group')
plt.show()

plt.scatter(df_final['tenure'], df_final['basket_count'], alpha=0.5)
plt.title('Tenure vs Shopping Count')
plt.show()

## 6. Time Analysis

In [ ]:
df_final['basket_date'] = pd.to_datetime(df_final['basket_date'])
df_final['Month'] = df_final['basket_date'].dt.month_name()
monthly_sales = df_final.groupby('Month')['basket_count'].sum().sort_values(ascending=False)
print(monthly_sales)

## 7. Customer Segmentation

In [ ]:
customer_summary = df_final.groupby('customer_id').agg({
    'basket_count':'sum',
    'tenure':'max',
    'customer_age':'first'
}).reset_index()

customer_summary.columns = ['CustomerID','TotalItems','MaxTenure','Age']

def classify_customer(row):
    if row['TotalItems'] >= 5:
        return 'VIP'
    elif row['TotalItems'] >= 3:
        return 'Regular'
    return 'New/Occasional'

customer_summary['CustomerType'] = customer_summary.apply(classify_customer, axis=1)
print(customer_summary.head())
print(customer_summary['CustomerType'].value_counts())

## 8. Feature Engineering

In [ ]:
df_final['Efficiency_score'] = df_final['basket_count'] / df_final['tenure']
print(df_final[['customer_id','basket_count','tenure','Efficiency_score']].head())

## 9. Outlier Detection

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x=df_final['basket_count'])
plt.title('Outlier Detection in Basket Count')
plt.show()

## 10. Machine Learning Model

In [ ]:
X = df_final[['customer_age', 'tenure']]
y = df_final['basket_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

prediction = model.predict([[25, 50]])
print('Prediction:', prediction[0])

y_pred = model.predict(X_test)
print('R2 Score:', r2_score(y_test, y_pred))